In [2]:
%pip install telebot pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 8.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import torch
import telebot
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSequenceClassification

TELEGRAM_TOKEN = "8204836715:AAHlfFXzG2NOeTsW8UH1ZCFRXWp4_vYDzRs"
LLM_PATH = "FP-KCV/jawani-sealion-gatra-2-9b"
BERT_PATH = "FP-KCV/javanese-honorific-classifier"

print("Memuat model klasifikasi...")
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_PATH)
bert_model = AutoModelForSequenceClassification.from_pretrained(BERT_PATH).to("cuda")

print("Memuat model LLM...")
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_PATH)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_PATH,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

honorific_map = {0: "Ngoko Lugu", 1: "Ngoko Alus", 2: "Krama Lugu", 3: "Krama Inggil"}

bot = telebot.TeleBot(TELEGRAM_TOKEN)

@bot.message_handler(commands=['start', 'help'])
def send_welcome(message):
    bot.reply_to(message, "Sugeng rawuh! Kula Guru Basa Jawa AI. Wonten menapa ingkang saged kula biyantu?")

@bot.message_handler(func=lambda message: True)
def handle_message(message):
    user_query = message.text
    chat_id = message.chat.id
    
    bot.send_chat_action(chat_id, 'typing')

    b_inputs = bert_tokenizer(user_query, return_tensors="pt", truncation=True, max_length=128).to("cuda")
    with torch.no_grad():
        b_outputs = bert_model(**b_inputs)
    
    predicted_id = b_outputs.logits.argmax().item()
    label = honorific_map.get(predicted_id, "Ngoko Lugu")

    is_krama = "Krama" in label
    
    if is_krama:
        system_instruction = f"Siswa nggunakake basa {label}. Minangka Guru, langsung wangsuli pitakonane nganggo basa Krama Inggil."
    else:
        system_instruction = f"Siswa nggunakake basa {label}. Minangka Guru, paringi koreksi ukara kasebut dadi Krama Inggil dhisik, banjur ing ngisore wangsuli pitakonane."

    prompt = f"Instruksi: {system_instruction}\nSiswa: {user_query}\nGuru AI:"
    
    inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_model.device)
    
    try:
        with torch.no_grad():
            output_tokens = llm_model.generate(
                **inputs, 
                max_new_tokens=256,
                temperature=0.7,
                top_p=0.9,
                do_sample=True,
                pad_token_id=llm_tokenizer.eos_token_id
            )
        
        full_response = llm_tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        
        if "Guru AI:" in full_response:
            clean_response = full_response.split("Guru AI:")[-1].strip()
        else:
            clean_response = full_response.replace(prompt, "").strip()
            
        final_output = f"[{label}]\n\n{clean_response}"
        bot.reply_to(message, final_output)
        
    except Exception as e:
        print(f"Error nalika generate response: {e}")
        bot.reply_to(message, "Nyuwun sewu, wonten gangguan teknis nalika ngolah pesen.")

print("Bot Telegram siap menerima pesan (Polling Mode Aktif)...")
bot.infinity_polling()

Memuat model ke VRAM... (Pastikan nggunakake GPU T4 x2)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/325 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/186 [00:00<?, ?B/s]

Bot Telegram siap menerima pesan (Polling Mode Aktif)...
